# Notebook 05 â€” Speech Pipeline

Wake Word (openWakeWord), Whisper STT, Piper TTS. All run on CPU on the Jetson.

## Cell 05-01 | Mount & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess

BASE   = '/content/drive/MyDrive/ARGUS'
MODELS = f'{BASE}/models/speech'
os.makedirs(MODELS, exist_ok=True)

# Install portaudio system lib first (needed by sounddevice)
subprocess.run(['apt-get', 'install', '-y', '-q', 'portaudio19-dev'], check=False)

# pyaudio removed — needs portaudio headers & build tools; sounddevice covers all use-cases
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'openwakeword', 'faster-whisper', 'sounddevice', 'soundfile', 'librosa'])
print('Speech tools installed')


## Cell 05-02 | Train Custom Wake Word 'ARGUS'

In [ ]:
import os

WAKE_DIR = f'{BASE}/datasets/wakeword'
os.makedirs(f'{WAKE_DIR}/positive', exist_ok=True)
os.makedirs(f'{WAKE_DIR}/negative', exist_ok=True)

print('WAKE WORD RECORDING INSTRUCTIONS')
print('Option A: Record in Colab (Cell 05-03)')
print(f'Option B: Upload WAV files to: {WAKE_DIR}/positive/')
print('  Each file: say ARGUS clearly, 1-2 seconds, WAV format')
print('  Target: 200 recordings')
print()
pos_dir = f'{WAKE_DIR}/positive'
n_pos   = len([f for f in os.listdir(pos_dir) if f.endswith('.wav')]) if os.path.exists(pos_dir) else 0
print(f'Current positive recordings: {n_pos} / 200 target')

## Cell 05-03 | Record Wake Word in Colab

In [ ]:
import sounddevice as sd
import soundfile as sf
import numpy as np, os, glob

WAKE_POS    = f'{BASE}/datasets/wakeword/positive'
os.makedirs(WAKE_POS, exist_ok=True)
SAMPLE_RATE = 16000
DURATION    = 2.0

# Recording only works when a real microphone is connected.
# In Colab headless (nbconvert) this is skipped gracefully.
try:
    devices = sd.query_devices()
    input_devs = [d for d in devices if d['max_input_channels'] > 0]
    if not input_devs:
        raise RuntimeError('No input devices found')
    print('Recording in 2 seconds... Say ARGUS clearly!')
    import time; time.sleep(2)
    recording = sd.rec(int(DURATION * SAMPLE_RATE),
                       samplerate=SAMPLE_RATE, channels=1, dtype='float32')
    sd.wait()
    n_existing = len(glob.glob(f'{WAKE_POS}/*.wav'))
    out_path   = f'{WAKE_POS}/argus_{n_existing:04d}.wav'
    sf.write(out_path, recording, SAMPLE_RATE)
    print(f'Saved: {out_path} | Total: {n_existing + 1}')
except Exception as e:
    print(f'No microphone available in this environment ({e})')
    print('To add wake word recordings: upload WAV files to:')
    print(f'  {WAKE_POS}')
    print('Then re-run Cell 05-04 to train the wake word model.')


## Cell 05-04 | Train Wake Word Model

In [ ]:
import os

WAKE_POS = f'{BASE}/datasets/wakeword/positive'
WAKE_OUT = f'{MODELS}/wakeword_argus'
TFLITE   = f'{WAKE_OUT}/argus.tflite'

n_pos = len([f for f in os.listdir(WAKE_POS) if f.endswith('.wav')]) if os.path.exists(WAKE_POS) else 0

if os.path.exists(TFLITE):
    size_kb = os.path.getsize(TFLITE) / 1024
    print(f'Wake word model already trained ({size_kb:.0f} KB) — skipping.')
    print(f'Model: {TFLITE}')
elif n_pos < 50:
    print(f'Only {n_pos} recordings found. Need at least 50 (200 recommended).')
    print('Record more using Cell 05-03 before training.')
else:
    print(f'{n_pos} recordings found — starting openWakeWord training...')
    from openwakeword.train import train_model
    os.makedirs(WAKE_OUT, exist_ok=True)
    train_model(
        model_name='argus',
        positive_dir=WAKE_POS,
        output_dir=WAKE_OUT,
        n_epochs=100,
        batch_size=64,
        learning_rate=0.001,
        generate_negative_examples=True,
    )
    if os.path.exists(TFLITE):
        print(f'Wake word model saved: {TFLITE}')
        print('On Jetson: from openwakeword.model import Model')
        print(f'  model = Model(wakeword_models=["{TFLITE}"])')
    else:
        print('WARNING: training finished but .tflite not found — check WAKE_OUT path.')


## Cell 05-05 | Setup Whisper STT

In [ ]:
from faster_whisper import WhisperModel
import os, numpy as np, soundfile as sf

WHISPER_DIR = f'{MODELS}/whisper'
os.makedirs(WHISPER_DIR, exist_ok=True)

print('Downloading Whisper-tiny INT8 quantized...')
model_stt = WhisperModel('tiny', device='cpu', compute_type='int8', download_root=WHISPER_DIR)
print('Whisper-tiny INT8 loaded (~39MB)')

dummy_audio = np.zeros(3 * 16000, dtype=np.float32)
sf.write('/tmp/test_audio.wav', dummy_audio, 16000)
segments, info = model_stt.transcribe('/tmp/test_audio.wav', language='en')
print(f'Whisper working | Language: {info.language}')
print('(silence gives empty transcription - correct)')

with open(f'{MODELS}/whisper_path.txt', 'w') as f: f.write(WHISPER_DIR)
print(f'Whisper model cached at: {WHISPER_DIR}')

## Cell 05-06 | Setup Piper TTS

In [ ]:
# hf_hub_download uses HF's cache — partial downloads are resumed automatically.
# We add a file-size guard so a truncated .onnx is caught and re-downloaded.
from huggingface_hub import hf_hub_download
import os

TTS_DIR  = f'{MODELS}/piper'
os.makedirs(TTS_DIR, exist_ok=True)

REPO     = 'rhasspy/piper-voices'
FILES    = [
    ('en/en_US/lessac/medium/en_US-lessac-medium.onnx',      50),  # ~60 MB
    ('en/en_US/lessac/medium/en_US-lessac-medium.onnx.json',  0),  # tiny config
]

for hf_path, min_mb in FILES:
    fname = os.path.basename(hf_path)
    dest  = f'{TTS_DIR}/{fname}'
    if os.path.exists(dest) and os.path.getsize(dest)/1e6 >= min_mb:
        print(f'✓ {fname} ({os.path.getsize(dest)/1e6:.1f} MB)')
        continue
    if os.path.exists(dest):
        print(f'Partial {fname} detected — removing for clean download')
        os.remove(dest)
    print(f'Downloading {fname} ...')
    local = hf_hub_download(repo_id=REPO, filename=hf_path, local_dir=TTS_DIR)
    print(f'✓ {fname} saved')

TTS_MODEL  = f'{TTS_DIR}/en_US-lessac-medium.onnx'
TTS_CONFIG = f'{TTS_DIR}/en_US-lessac-medium.onnx.json'
print(f'✅ Piper TTS ready')